In [1]:
import json, gc
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict

from src.minbpe import RegexTokenizer
from src.gpt import GPTLanguageModel
from src.lora import get_lora_model, print_trainable_parameters

import torch
torch.set_float32_matmul_precision('high')
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

class FineTuningDataset(Dataset):
    def __init__(self, data: torch.Tensor, device: torch.device, padding_token: int):
        self.data = data
        self.device = device
        self.padding_token = padding_token

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sample = self.data[index]
        x = sample.to(self.device)
        y = sample[1:].to(self.device)
        padding_tensor = torch.tensor([self.padding_token], device=self.device)
        y = torch.cat((y, padding_tensor))

        return x, y

@torch.no_grad()
def estimate_loss(
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
) -> Dict[str, float]:
    output = {}
    model.eval()

    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = []
        for x, y in loader:
            with torch.no_grad():
                _, loss = model(x, y)
            losses.append(loss.item())
        output[split] = sum(losses) / len(losses)

    model.train()
    return output

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"

checkpoint_dir = Path("data") / "ch07_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [4]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_tensor = torch.load(tokenizer_dir /'ch05_train.ft.pt')
val_tensor = torch.load(tokenizer_dir /'ch05_validation.ft.pt')

In [5]:
#block_size = 256 # ch02: 256, ch03: 512
#n_embd = 512
#n_head = 8
#n_layer = 4
#dropout = 0.2
#vocab_size = len(tokenizer.vocab)

padding_token = -100
learning_rate = 1e-3
lora_config = {
    "rank": 4,
    "alpha": 8,
}

batch_size = 64
iter_start, max_iters = 1, 100
eval_interval = 50

In [6]:
bm_path = Path("data") / "ch02_checkpoints" / "checkpoint_001-377870.pt"

bm_ckpt = torch.load(bm_path, weights_only=True, map_location=device)
parameters = bm_ckpt['meta']['parameters']

base_model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

base_model = torch.compile(base_model)
base_model.load_state_dict(bm_ckpt["model_state_dict"])

num_parameters = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"load base model: {bm_path}, {num_parameters:_.3}M parameters")

load base model: data/ch02_checkpoints/checkpoint_001-377870.pt, 13.8M parameters


In [10]:
model = get_lora_model(
    model=base_model,
    lora_config=lora_config,
    device=device,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

ckpt_files = sorted(
    checkpoint_dir.glob("checkpoint_*.pt"),
    #key=lambda x: x.stat().st_ctime,
    key=lambda x: int(x.name.replace("checkpoint_", "").replace(".pt", "")),
    reverse=True,
)
#print(ckpt_files)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    iter_start = int(checkpoint_path.name.replace("checkpoint_", "").replace(".pt", "")) + 1
    print(f"load previous model checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print(f"learning rate: {optimizer.param_groups[0]['lr']}")
print_trainable_parameters(model)

load previous model checkpoint: data/ch07_checkpoints/checkpoint_000069.pt
learning rate: 0.0001
All parameters: 14.12M | Trainable parameters: 0.33M | Trainable %: 2.31%


In [11]:
input_tokens = tokenizer.encode("hello, world", allowed_special="all")
input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)
    answer = tokenizer.decode(output[0].tolist())
    print(f"--> lora_model:\n{answer}", )

#del pre_model
#gc.collect()
#torch.cuda.empty_cache()

#print(model)

--> lora_model:
hello, world, global market notation)

It is also worth noting that this is a multi-dimensional example. 

I hope this answers the question was helpful. The time complexity theory and current state order have been referenced to have insights into the potential fine-tuning analysis and other fields


In [12]:
train_dataset = FineTuningDataset(data=train_tensor, device=device, padding_token=padding_token)
val_dataset = FineTuningDataset(data=val_tensor, device=device, padding_token=padding_token)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size)

train_losses, val_losses = [], []

In [ ]:
total_steps = len(train_loader)
#print(f"--> Start training: iter_start={iter_start}, max_iters={max_iters}, total_steps={total_steps}")

def _estimate_loss(iteration, step):
    losses = estimate_loss(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
    )

    train_losses.append(losses['train'])
    val_losses.append(losses['val'])

    print(f"{now()} iteration={iteration:03}/{max_iters:03}, step={step:07_}/{total_steps:07_}, ", end='')
    print(f"train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}")


_estimate_loss(iter_start, 0)

for iteration in range(iter_start, max_iters+1):
    ####
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        step = batch_idx + 1

        # Training step
        logits, loss = model(x_batch, y_batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        #batch_loss = loss.item()
        #print(f"batch_loss: {batch_loss}")

        # Evaluation
        if step % eval_interval == 0 or step == len(train_loader):
            _estimate_loss(iteration, step)

    ####
    checkpoint_prefix = str(checkpoint_dir / f"checkpoint_{iteration:06}")

    losses = estimate_loss(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
    )

    meta = {
        'created_at': now(),
        'lora_config': lora_config,
        "epoch": iteration,
        'train_loss': float(losses['train']),
        'validation_loss': float(losses['val']),
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    with open(checkpoint_prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    torch.save(checkpoint, checkpoint_prefix+".pt")
    print(f"{now()} saved checkpoint: {checkpoint_prefix}.pt")

2025-09-05T13:39:07+08:00 iteration=070/100, step=000_000/000_493, train_loss=3.626, validation_loss=4.685
2025-09-05T13:40:13+08:00 iteration=070/100, step=000_050/000_493, train_loss=3.401, validation_loss=4.461
2025-09-05T13:40:54+08:00 iteration=070/100, step=000_100/000_493, train_loss=3.355, validation_loss=4.415
2025-09-05T13:41:35+08:00 iteration=070/100, step=000_150/000_493, train_loss=3.356, validation_loss=4.432
2025-09-05T13:42:17+08:00 iteration=070/100, step=000_200/000_493, train_loss=3.292, validation_loss=4.347
2025-09-05T13:42:58+08:00 iteration=070/100, step=000_250/000_493, train_loss=3.338, validation_loss=4.396
2025-09-05T13:43:40+08:00 iteration=070/100, step=000_300/000_493, train_loss=3.285, validation_loss=4.339


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()